# Geo Spatial Data Extraction

In [1]:
import os
import json
import pandas as pd

path = r"D:/Viki/Guvi DS/Capstone Projects/PhonePe-Pulse-Project/pulse/data/map/insurance/country/india/state"

data = []

for state in os.listdir(path):
    state_path = os.path.join(path, state)

    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)

        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter = int(file.strip(".json"))
                file_path = os.path.join(year_path, file)

                with open(file_path, "r") as f:
                    content = json.load(f)

                try:
                    records = content["data"]["data"]["data"]

                    for item in records:
                        data.append({
                            "State": state,
                            "Year": int(year),
                            "Quarter": quarter,
                            "District": item[3],
                            "Latitude": item[0],
                            "Longitude": item[1],
                            "Metric": item[2]
                        })

                except:
                    pass

df = pd.DataFrame(data)

#--------------------------------------------------------------------------------------------------

# Display data
print("Extracted Data:")
print(df.head())

print("\nColumns:")
print(df.columns)

print("\nTotal Rows:", df.shape[0])

# Save CSV
df.to_csv("map_insurance_geo.csv", index=False)

print("\nCSV 'map_insurance_geo.csv' created successfully!")

Extracted Data:
                       State  Year  Quarter                District  \
0  andaman-&-nicobar-islands  2020        2       nicobars district   
1  andaman-&-nicobar-islands  2020        2  south andaman district   
2  andaman-&-nicobar-islands  2020        2  south andaman district   
3  andaman-&-nicobar-islands  2020        3  south andaman district   
4  andaman-&-nicobar-islands  2020        3  south andaman district   

    Latitude  Longitude  Metric  
0   9.173490  92.812846     3.0  
1  11.665257  92.753094     2.0  
2  11.665446  92.733193     1.0  
3  11.653981  92.743142     8.0  
4  11.654075  92.723246     8.0  

Columns:
Index(['State', 'Year', 'Quarter', 'District', 'Latitude', 'Longitude',
       'Metric'],
      dtype='object')

Total Rows: 1043137

CSV 'map_insurance_geo.csv' created successfully!


# Geo Spatial Data Table Creation

In [2]:
import pandas as pd
import mysql.connector

# Load CSV (generated from your extraction code)
df = pd.read_csv("map_insurance_geo.csv")

# Connect to MySQL
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="12345678",
    database="phonepe_db"
)

cursor = connection.cursor()

# Drop table if exists
cursor.execute("DROP TABLE IF EXISTS map_insurance_geo")

# Create table
cursor.execute("""
CREATE TABLE map_insurance_geo (
    State VARCHAR(100),
    Year INT,
    Quarter INT,
    District VARCHAR(150),
    Latitude DOUBLE,
    Longitude DOUBLE,
    Metric DOUBLE
)
""")

print("Table 'map_insurance_geo' created successfully!")

# Display table structure
cursor.execute("DESCRIBE map_insurance_geo")
print("\nTable Structure:")
for row in cursor.fetchall():
    print(row)

# Close connection
cursor.close()
connection.close()

Table 'map_insurance_geo' created successfully!

Table Structure:
('State', 'varchar(100)', 'YES', '', None, '')
('Year', 'int', 'YES', '', None, '')
('Quarter', 'int', 'YES', '', None, '')
('District', 'varchar(150)', 'YES', '', None, '')
('Latitude', 'double', 'YES', '', None, '')
('Longitude', 'double', 'YES', '', None, '')
('Metric', 'double', 'YES', '', None, '')


In [3]:
import pandas as pd
import mysql.connector

# Load CSV
df = pd.read_csv("map_insurance_geo.csv")

# Connect to MySQL
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="12345678",
    database="phonepe_db"
)

cursor = connection.cursor()

# Insert query
insert_query = """
INSERT INTO map_insurance_geo (
    State, Year, Quarter, District, Latitude, Longitude, Metric
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

# Insert row by row
for i, row in df.iterrows():
    cursor.execute(insert_query, tuple(row))

connection.commit()

print("Data inserted successfully!")

# Verify
cursor.execute("SELECT COUNT(*) FROM map_insurance_geo")
print("Total Rows:", cursor.fetchone()[0])

# Close connection
cursor.close()
connection.close()

Data inserted successfully!
Total Rows: 1043137
